In [ ]:
import swissparlpy as spp
import logging
import pandas as pd
from typing import cast

from pprint import pprint
import yaml

from datetime import datetime

In [ ]:
loglevel = logging.DEBUG
logging.basicConfig(
    format="%(asctime)s %(name)s %(levelname)-8s %(message)s",
    level=loglevel,
    datefmt="%Y-%m-%d %H:%M:%S",
)
logging.captureWarnings(True)

In [ ]:
def dump(d):
    print(yaml.dump(d, allow_unicode=True, default_flow_style=False))

In [ ]:
# Kantonsrat Zürich
tables = spp.get_tables(backend="gever_canton_zurich")
pprint(tables)

In [ ]:
# City of Zurich (Gemeinderat)
tables = spp.get_tables(backend="gever_city_zurich")
pprint(tables)

In [ ]:
canton_zh_client = spp.SwissParlClient(backend="gever_canton_zurich")
city_zh_client = spp.SwissParlClient(backend="gever_city_zurich")

# Kantonsrat ZH Übersicht über alle Tabellen und Spalten

In [ ]:
overview = canton_zh_client.get_overview()
for table, props in overview.items():
    print(table)
    for prop in props:
        print(f" + {prop}")
    print("")

# Gemeinderat Stadt Zurich Übersicht über alle Tabellen und Spalten

In [ ]:
overview = city_zh_client.get_overview()
for table, props in overview.items():
    print(table)
    for prop in props:
        print(f" + {prop}")
    print("")

In [ ]:
res = city_zh_client.get_data("Dokument")
res

In [ ]:
res[0]

## Fraktionen im Kantonsrat

In [ ]:
factions = canton_zh_client.get_data("Behoerden", query="GremiumTyp adj Fraktion")
factions.to_dataframe()

In [ ]:
for faction in factions:
    print(f"Fraktion: {faction['name']}")
    dump(faction)

    # get members of faction
    print("Mitglieder:")
    members = canton_zh_client.get_data(
        "Mitglieder",
        query=f"dauer_end >= \"2020-11-19 00:00:00\" and dauer_start <= \"2020-11-19 00:00:00\" and gremium adj {faction['kurzname']} sortBy name/sort.ascending vorname/sort.ascending",
    )

    name_members = [f"{m['name']}, {m['vorname'] or ''} ({m['ort']})" for m in members]
    dump(name_members)

In [ ]:
member_dfs = []
today = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
for faction in factions:
    members = canton_zh_client.get_data(
        "Mitglieder",
        query=f"dauer_end >= \"{today}\" and dauer_start <= \"{today}\" and gremium adj {faction['kurzname']} sortBy name/sort.ascending vorname/sort.ascending",
    )

    member_dfs.append(members.to_dataframe())
df_member = pd.concat(member_dfs).reset_index(drop=True)
df_member

In [ ]:
pprint(canton_zh_client.get_data("Mitglieder")[1])

In [ ]:
from flatten_dict import flatten

In [ ]:
pprint(flatten(members[0], "underscore"))

In [ ]:
members[0]

In [ ]:
pd.DataFrame(members[0:5]).columns

In [ ]:
members.to_dataframe().columns

In [ ]:
members_df = members.to_dataframe()
members_df

In [ ]:
canton_zh_client.get_variables("Mitglieder")